In [4]:
"""
End-to-end pipeline: CD133 E14 vs E18
DESeq2 -> Consensus Peaks -> Enhancer Integration -> CellOracle Base GRN -> GRN Pruning
"""

import os
import pandas as pd
import numpy as np
from importlib import reload
import pandas as pd 
import numpy as np 
import sys
import os
# get project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if in notebook:
# project_root = os.path.abspath('..')   # or adjust as needed
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.pipeline as pipeline
import src.enhancer_atac as enatac
import src.grn_pruner as grnpruner
# Reload modules
reload(pipeline)
reload(enatac)
reload(grnpruner)

# ==============================================================================
# PATHS
# ==============================================================================

base_dir = "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC"

paths = {
    # RNA-seq
    "counts": os.path.join(base_dir, "data/rna_seq/E14_ E18_LGE_cortex_seq_Counts.csv"),
    "metadata": os.path.join(base_dir, "data/rna_seq/Samples+Pooling_RNA-Seq.csv"),
    
    # ATAC-seq
    "atac_metadata": os.path.join(base_dir, "data/atac_seq/ATAC-seq/samples_clean.csv"),
    "atac_peak_dir": os.path.join(base_dir, "data/atac_seq/ATAC-seq"),
    
    # Annotations
    "tf_list": os.path.join(base_dir, "data/annotations/Mouse_TFs_Kinases_webpage-3-30-2017.xlsx"),
    "gtf": os.path.join(base_dir, "data/annotations/gencode.vM36.annotation.gtf"),
    "enhancer_files": [
        os.path.join(base_dir, "data/annotations/enhancerAtlas_neuron_cortical.txt"),
        os.path.join(base_dir, "data/annotations/enhancerAtlas_brain_e14.5.txt"),
        os.path.join(base_dir, "data/annotations/enhancerAtlas_cortex.txt")
    ],
    
    # Integrated data (for pipeline object only)
    "overlap_df": os.path.join(base_dir, "data/integrated/overlap_annotated.tsv"),
    "chip_annotated": os.path.join(base_dir, "data/integrated/chip_annotated_filtered.tsv"),
    "atac_annotated": os.path.join(base_dir, "data/integrated/atac_annotated.tsv"),
    
    # Output paths
    "output_dir": os.path.join(base_dir, "results/cd133_e14_e18_grn"),
    "consensus_bed": os.path.join(base_dir, "results/cd133_e14_e18_grn/consensus_cd133.bed"),
    "deseq_output": os.path.join(base_dir, "results/cd133_e14_e18_grn/deseq_temporal_cd133.csv"),
    "celloracle_h5": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_tfinfo.h5"),
    "celloracle_parquet": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_base_grn.parquet"),
    "e14_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E14_TF_network.csv"),
    "e18_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E18_TF_network.csv")
}

# Create output directory
os.makedirs(paths["output_dir"], exist_ok=True)

exclude_samples = ['MUC9939', 'MUC9940', 'MUC9914']

print("="*80)
print("CD133 E14 vs E18 GRN PIPELINE")
print("="*80)

# ==============================================================================
# STEP 1: RNA-SEQ DIFFERENTIAL EXPRESSION (CD133 E14 vs E18)
# ==============================================================================

print("\n" + "="*80)
print("STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS")
print("="*80)

nsc = pipeline.NSCAnalysis(
    counts_path=paths['counts'],
    metadata_path=paths['metadata'],
    atac_metadata_path=paths['atac_metadata'],
    overlap_df_path=paths['overlap_df'],
    chip_annotated_path=paths['chip_annotated'],
    atac_annotated_path=paths['atac_annotated'],
    tf_list_path=paths['tf_list'],
    gtf_path=paths['gtf'],
    exclude_samples=exclude_samples
)

# Filter for CD133 only
cd133_samples = nsc.metadata[nsc.metadata['Marker'] == 'Progenitors'].index.tolist()
nsc.counts = nsc.counts[[c for c in cd133_samples if c in nsc.counts.columns]]
nsc.metadata = nsc.metadata.loc[cd133_samples]

print(f"\nCD133 samples: {len(nsc.metadata)}")
print(nsc.metadata[['Stage', 'Region', 'Marker']].value_counts())

# Run DESeq2
deseq_results = nsc.run_deseq(group1='E14', group2='E18', group_col='Stage')

# Save DESeq results
deseq_results.to_csv(paths['deseq_output'])
print(f"\nDESeq results saved to: {paths['deseq_output']}")

# Get DE TFs
de_tfs = deseq_results[
    (deseq_results['padj'] < 0.05) & 
    (deseq_results['log2FoldChange'].abs() > 1) &
    (deseq_results['is_TF'] == True)
]

deg_list_e14 = de_tfs[de_tfs['log2FoldChange'] > 1]['symbol'].tolist()
deg_list_e18 = de_tfs[de_tfs['log2FoldChange'] < -1]['symbol'].tolist()

print(f"\nE14-high TFs: {len(deg_list_e14)}")
print(f"E18-high TFs: {len(deg_list_e18)}")
print(f"\nE14 TFs: {deg_list_e14[:10]}...")
print(f"E18 TFs: {deg_list_e18[:10]}...")

# ==============================================================================
# STEP 2: BUILD CONSENSUS PEAKS FROM ATAC-SEQ
# ==============================================================================

print("\n" + "="*80)
print("STEP 2: CONSENSUS PEAK BUILDING")
print("="*80)

builder = enatac.ConsensusPeakBuilder(
    metadata_file=paths['atac_metadata'],
    peak_dir=paths['atac_peak_dir'],
    factor='CD133',
    conditions=['E14', 'E18']
)

consensus = builder.build_consensus()
builder.save_bed(consensus, paths['consensus_bed'])

print(f"\nConsensus peaks: {len(consensus)}")
print(f"Saved to: {paths['consensus_bed']}")

# ==============================================================================
# STEP 3: ENHANCER INTEGRATION
# ==============================================================================

print("\n" + "="*80)
print("STEP 3: ENHANCER ATLAS INTEGRATION")
print("="*80)

integrator = enatac.EnhancerIntegrator(
    enhancer_files=paths['enhancer_files'],
    from_assembly='mm9'
)

enhancers_mm39 = integrator.load_and_liftover()

# Overlap with consensus peaks
target_genes, overlaps_df = integrator.overlap_with_consensus(consensus)

# Get DE TFs with accessible enhancers
e14_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e14)
e18_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e18)

print(f"\nE14 TFs with accessible enhancers: {e14_tfs_with_enh}")
print(f"E18 TFs with accessible enhancers: {e18_tfs_with_enh}")

# Save enhancer overlaps
overlaps_df.to_csv(os.path.join(paths['output_dir'], 'enhancer_consensus_overlaps.csv'), index=False)


CD133 E14 vs E18 GRN PIPELINE

STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS
Loading data...

[1/9] Loading counts...
      Dropping 1 genes with NaN values
      28664 genes x 36 samples

[2/9] Loading RNA-seq metadata...
      36 samples
      Columns: ['ForeignID', 'Stage', 'Replicate', 'Region', 'Marker', 'Shuffle']

[3/9] Loading ATAC-seq metadata...
      15 samples
      Columns: ['Tissue', 'Factor', 'Condition', 'Treatment', 'Replicate', 'bamReads', 'Peaks', 'PeakCaller']

[4/9] Creating ATAC-to-RNA sample mapping...
      Mapped: 15 / 15 ATAC samples

[5/9] Removing outlier samples...
      Removed: ['MUC9939', 'MUC9940', 'MUC9914']
      Remaining: 33 samples

[6/9] Filtering low-count genes...

Filtering low-count genes
Threshold: counts >= 10 in >= 3 samples
  E14: 17483 genes pass filter
  E18: 17939 genes pass filter

Genes before: 28664
Genes after:  18187
Removed:      10477 (36.6%)

[7/9] Loading overlap_df (ChIP ∩ ATAC)...
      448802 overlaps
      Columns: ['chip_chr', 

Fitting size factors...
... done in 0.03 seconds.

Fitting dispersions...
... done in 27.16 seconds.

Fitting dispersion trend curve...
... done in 0.78 seconds.

Fitting MAP dispersions...
... done in 43.68 seconds.

Fitting LFCs...
... done in 13.26 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 14 outlier genes.

Fitting dispersions...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.02 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Running Wald tests...


Extracting results (E14 vs E18)...


... done in 4.27 seconds.



Log2 fold change & Wald test p-value: Stage E14 vs E18
                        baseMean  log2FoldChange     lfcSE      stat  \
ENSMUSG00000000001  11392.735011        0.163045  0.071729  2.273084   
ENSMUSG00000000088   7290.014219        0.374888  0.087763  4.271573   
ENSMUSG00000000581   2662.931320       -0.024108  0.077363 -0.311623   
ENSMUSG00000006471   2191.953433        0.199988  0.038724  5.164521   
ENSMUSG00000036019    465.437536       -0.415655  0.145974 -2.847467   
...                          ...             ...       ...       ...   
ENSMUSG00000035984    171.993688       -4.000665  0.435222 -9.192241   
ENSMUSG00000035992   1390.075479       -0.098378  0.124280 -0.791582   
ENSMUSG00000036002   2360.311704       -0.017796  0.059961 -0.296788   
ENSMUSG00000036006    606.543527       -1.622156  0.212070 -7.649142   
ENSMUSG00000036009    659.660379       -0.053090  0.080047 -0.663239   

                          pvalue          padj  
ENSMUSG00000000001  2.302114e-0

/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:657: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_sig_tf = results[sig_mask & results['is_TF']].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:658: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_sig_gene = results[sig_mask & ~results['is_TF']].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:660: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_up = results[sig_mask & (results['log2FoldChange'] > 0)].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:661: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_down = results[sig_mask & (results['log2FoldChange'] < 0)].shape[0]



DESeq results saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/deseq_temporal_cd133.csv

E14-high TFs: 39
E18-high TFs: 83

E14 TFs: ['Hmga2', 'Smad3', 'Rcor2', 'Plagl2', 'Sox3', 'Sall4', 'Lhx9', 'Prdm12', 'Lef1', 'Hmga1']...
E18 TFs: ['Csdc2', 'Creb3l2', 'Egr1', 'Etv4', 'Dbx2', 'Klf9', 'Nfatc1', 'Sox10', 'Foxo1', 'Zbtb7c']...

STEP 2: CONSENSUS PEAK BUILDING
Found 10 samples for CD133 in ['E14', 'E18']
Consensus peaks: 93492
Saved to /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/consensus_cd133.bed

Consensus peaks: 93492
Saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/consensus_cd133.bed

STEP 3: ENHANCER ATLAS INTEGRATION
Initializing liftOver from mm9 to mm39...
Loading enhancerAtlas_neuron_cortical.txt
Loading enhancerAtlas_brain_e14.5.txt
Loading enhancerAtlas_cortex.txt
Loaded 180809 unique enhancer-gene pairs in mm9
Lifting over to mm3

In [8]:
import src.base_grn as base_grn

In [ ]:
# ==============================================================================
# STEP 4: CELLORACLE BASE GRN CONSTRUCTION
# ==============================================================================

print("\n" + "="*80)
print("STEP 4: CELLORACLE BASE GRN CONSTRUCTION")
print("="*80)


# Initialize CellOracle
co = base_grn.GRNCo(
    bed_path=paths['consensus_bed'],
    ref_genome='mm39',
    genomes_dir=None
)

# Step 1: Load BED
print("\nLoading consensus peaks...")
co.load_bed()

# Step 2: Annotate TSS
print("Annotating TSS...")
co.annotate_tss()

# Step 3: Ensure genome
print("Checking genome installation...")
co.ensure_genome()

# Step 4: Scan motifs
print("Scanning TF motifs (this takes time)...")
co.scan_motifs(fpr=0.02, verbose=True)

# Step 5: Filter motifs
print("Filtering motifs...")
co.filter_motifs(score_threshold=10)




STEP 4: CELLORACLE BASE GRN CONSTRUCTION

Loading consensus peaks...
Annotating TSS...
que bed peaks: 93492
tss peaks in que: 36221
Checking genome installation...
Scanning TF motifs (this takes time)...
No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-12-15 16:35:40,326 - DEBUG - using background: genome mm39 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 

Motif scan started .. It may take long time.



Scanning:   0%|          | 0/17436 [00:00<?, ? sequences/s]

Filtering motifs...
Filtering finished: 9877109 -> 1839134
1. Converting scanned results into one-hot encoded dataframe.


...

2. Converting results into dictionaries.


  0%|          | 0/18608 [00:00<?, ?it/s]

  0%|          | 0/1090 [00:00<?, ?it/s]

Saving CellOracle outputs...


ValueError: Filename needs to end with '.celloracle.tfinfo'

In [38]:
reload(grnpruner)
# Step 6: Save outputs
print("Saving CellOracle outputs...")
co.save_tfinfo(paths['celloracle_h5'])
base_grn_df = co.save_dataframe(paths['celloracle_parquet'])

print(f"\nBase GRN saved to: {paths['celloracle_parquet']}")
print(f"Base GRN shape: {base_grn_df.shape}")
print(f"TFs in base GRN: {base_grn_df.shape[1] - 2}")  # -2 for peak_id and gene columns

# ==============================================================================
# STEP 5: GRN PRUNING (CORRELATION + DE FILTERING)
# ==============================================================================

print("\n" + "="*80)
print("STEP 5: GRN PRUNING")
print("="*80)

# Get all TFs (E14 + E18 DE TFs)
all_tf_list = list(set(deg_list_e14 + deg_list_e18))
print(f"\nTotal DE TFs for pruning: {len(all_tf_list)}")

# Initialize pruner
pruner = grnpruner.GRNPruner(
    pkn_file=paths['celloracle_parquet'],
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    tf_list=all_tf_list,
    lfc_threshold=1.0,
    corr_threshold=0.3,
    qval_threshold=0.05
)

# Build stage-specific networks
e14_grn, e18_grn = pruner.build_networks()

# Save networks
e14_grn.to_csv(paths['e14_grn'], index=False)
e18_grn.to_csv(paths['e18_grn'], index=False)

print(f"\nE14 network saved to: {paths['e14_grn']}")
print(f"E18 network saved to: {paths['e18_grn']}")

# ==============================================================================
# STEP 6: SUMMARY
# ==============================================================================

print("\n" + "="*80)
print("PIPELINE COMPLETE - SUMMARY")
print("="*80)

summary = pd.DataFrame({
    'Step': [
        '1. DESeq2',
        '2. Consensus Peaks',
        '3. Enhancer Integration',
        '4. CellOracle Base GRN',
        '5. E14 TF Network',
        '6. E18 TF Network'
    ],
    'Count': [
        f"{len(de_tfs)} DE TFs",
        f"{len(consensus)} peaks",
        f"{len(target_genes)} genes with accessible enhancers",
        f"{base_grn_df.shape[1] - 2} TFs in base GRN",
        f"{len(e14_grn)} edges, {e14_grn['TF'].nunique()} TFs",
        f"{len(e18_grn)} edges, {e18_grn['TF'].nunique()} TFs"
    ]
})

print(summary.to_markdown(index=False))

print("\n" + "="*80)
print("OUTPUT FILES:")
print("="*80)
for key, path in paths.items():
    if 'output' in key or any(x in key for x in ['consensus', 'deseq', 'celloracle', 'grn']):
        print(f"  {key}: {path}")

print("\n" + "="*80)
print("NETWORK COMPARISON:")
print("="*80)

e14_edges = set(zip(e14_grn['TF'], e14_grn['target']))
e18_edges = set(zip(e18_grn['TF'], e18_grn['target']))

shared_edges = e14_edges & e18_edges
e14_specific = e14_edges - e18_edges
e18_specific = e18_edges - e14_edges

print(f"\nShared edges: {len(shared_edges)}")
print(f"E14-specific edges: {len(e14_specific)}")
print(f"E18-specific edges: {len(e18_specific)}")

print("\nTop 10 E14 hub TFs:")
print(e14_grn['TF'].value_counts().head(10))

print("\nTop 10 E18 hub TFs:")
print(e18_grn['TF'].value_counts().head(10))

Saving CellOracle outputs...

Base GRN saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/celloracle_base_grn.parquet
Base GRN shape: (20783, 1092)
TFs in base GRN: 1090

STEP 5: GRN PRUNING

Total DE TFs for pruning: 122
Loading data...
PKN: 20783 peaks, TF list: 122 TFs

=== Converting PKN ===
Total edges: 2729178

=== Filtering to TF-TF ===
TF-TF edges: 1986

=== E14 Network ===
After DE filter: 134
Correlations computed: 131
Final E14: 50 edges, 11 TFs

=== E18 Network ===
After DE filter: 1046
Correlations computed: 1034
Final E18: 59 edges, 18 TFs

E14 network saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/E14_TF_network.csv
E18 network saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/E18_TF_network.csv

PIPELINE COMPLETE - SUMMARY
| Step                    | Count                                 |
|:------------------------|:------

In [43]:
e14_grn

,TF,target,correlation,pval,qval
0,E2f2,Hmga2,0.842424,0.002220,0.009694
1,E2f3,Hmga2,0.684848,0.028883,0.042041
4,Nhlh1,Otx1,0.878788,0.000814,0.004265
5,Nhlh2,Otx1,0.721212,0.018573,0.030414
10,Bcl11b,Six4,0.745455,0.013330,0.024946
13,Nhlh1,Six4,0.781818,0.007547,0.018654
14,Nhlh2,Six4,0.830303,0.002940,0.011329
15,Nr0b1,Six4,0.660606,0.037588,0.049738
16,E2f2,E2f3,0.903030,0.000344,0.002501
23,Nhlh1,Neurog1,0.963636,0.000007,0.000320


In [44]:
e18_grn

,TF,target,correlation,pval,qval
1,Klf15,Gli1,0.854545,0.001637,0.033849
15,Tfap2d,Hivep2,0.963636,0.000007,0.002523
38,Ets1,Hey2,0.903030,0.000344,0.014804
47,Klf9,Hey2,0.842424,0.002220,0.039578
91,Ppara,Stat5a,0.830303,0.002940,0.040536
100,Nr4a1,Stat5a,0.806061,0.004862,0.048809
111,Fli1,Etv4,0.866667,0.001174,0.029596
113,Foxf2,Etv4,0.830303,0.002940,0.040536
115,Foxq1,Etv4,0.830303,0.002940,0.040536
134,Fos,Foxj1,0.939394,0.000055,0.005671


## Plot the GRN

In [39]:
def plot_hierarchical_network(grn_df, stage_name, output_file, 
                              break_cycles=True, figsize=(30, 20)):
    """
    Plot hierarchical TF network.
    
    Args:
        grn_df: DataFrame with columns [TF, target, correlation, ...]
        stage_name: Name for title (e.g., 'E14', 'E18')
        output_file: Path to save PNG
        break_cycles: If True, break cycles for true topological hierarchy
        figsize: Figure size
    """
    import networkx as nx
    import matplotlib.pyplot as plt
    
    # Remove self-loops
    grn_clean = grn_df[grn_df['TF'] != grn_df['target']].copy()
    print(f"\n{stage_name}: Removed {len(grn_df) - len(grn_clean)} self-loops")
    
    # Build graph
    G = nx.from_pandas_edgelist(grn_clean, source='TF', target='target', 
                                 create_using=nx.DiGraph())
    
    print(f"{stage_name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    # Break cycles if requested
    if break_cycles:
        G_plot = G.copy()
        removed = 0
        while not nx.is_directed_acyclic_graph(G_plot):
            try:
                cycle = nx.find_cycle(G_plot)
                G_plot.remove_edge(cycle[0][0], cycle[0][1])
                removed += 1
            except nx.NetworkXNoCycle:
                break
        print(f"{stage_name}: Removed {removed} edges to break cycles")
        layout_type = "True Topological Hierarchy"
    else:
        G_plot = G
        layout_type = "Pseudo-Hierarchy (by degree)"
    
    # Calculate layout
    plt.figure(figsize=figsize)
    
    try:
        # Try topological layout
        layers = {}
        for i, nodes in enumerate(nx.topological_generations(G_plot)):
            for node in nodes:
                layers[node] = i
        
        layer_nodes = {}
        for node, layer in layers.items():
            if layer not in layer_nodes:
                layer_nodes[layer] = []
            layer_nodes[layer].append(node)
        
        pos = {}
        for layer, nodes in layer_nodes.items():
            n = len(nodes)
            for i, node in enumerate(nodes):
                x = (i - n/2) * 2
                y = -layer * 3
                pos[node] = (x, y)
        
        print(f"{stage_name}: Using topological layout")
        
    except nx.NetworkXUnfeasible:
        # Fallback: pseudo-hierarchy
        print(f"{stage_name}: Still has cycles, using pseudo-hierarchy")
        in_degrees = dict(G_plot.in_degree())
        out_degrees = dict(G_plot.out_degree())
        
        layers = {}
        for node in G_plot.nodes():
            layers[node] = out_degrees[node] - in_degrees[node]
        
        max_score = max(layers.values()) if layers.values() else 1
        min_score = min(layers.values()) if layers.values() else 0
        
        for node in layers:
            if max_score != min_score:
                layers[node] = int(5 * (layers[node] - min_score) / (max_score - min_score))
            else:
                layers[node] = 0
        
        layer_nodes = {}
        for node, layer in layers.items():
            if layer not in layer_nodes:
                layer_nodes[layer] = []
            layer_nodes[layer].append(node)
        
        pos = {}
        for layer, nodes in layer_nodes.items():
            n = len(nodes)
            for i, node in enumerate(nodes):
                x = (i - n/2) * 2
                y = layer * 3
                pos[node] = (x, y)
        
        layout_type = "Pseudo-Hierarchy (by degree)"
    
    # Adjust font size based on network size
    if G_plot.number_of_nodes() > 100:
        font_size = 6
        node_size = 300
    elif G_plot.number_of_nodes() > 50:
        font_size = 8
        node_size = 500
    else:
        font_size = 10
        node_size = 800
    
    # Draw
    nx.draw_networkx_nodes(G_plot, pos, node_size=node_size, node_color='lightblue', alpha=0.8)
    nx.draw_networkx_edges(G_plot, pos, alpha=0.3, arrows=True, arrowsize=10, 
                           edge_color='gray', width=1)
    nx.draw_networkx_labels(G_plot, pos, font_size=font_size, font_weight='bold')
    
    title = f'{stage_name} TF Network - {layout_type}\n({G_plot.number_of_nodes()} nodes, {G_plot.number_of_edges()} edges)'
    plt.title(title, fontsize=18)
    plt.axis('off')
    plt.tight_layout()
    
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    
    print(f"{stage_name}: Saved to {output_file}\n")
    
    return G_plot


# Usage - full networks:
e14_graph = plot_hierarchical_network(
    grn_df=e14_grn,
    stage_name='E14',
    output_file='e14_network_full_hierarchical.png',
    break_cycles=True,
    figsize=(30, 20)  # Large canvas for full network
)

e18_graph = plot_hierarchical_network(
    grn_df=e18_grn,
    stage_name='E18',
    output_file='e18_network_full_hierarchical.png',
    break_cycles=True,
    figsize=(30, 20)
)


E14: Removed 0 self-loops
E14: 29 nodes, 50 edges
E14: Removed 1 edges to break cycles
E14: Using topological layout
E14: Saved to e14_network_full_hierarchical.png


E18: Removed 0 self-loops
E18: 37 nodes, 59 edges
E18: Removed 5 edges to break cycles
E18: Using topological layout
E18: Saved to e18_network_full_hierarchical.png



In [40]:
e14_grn['target'].unique()

array(['Hmga2', 'Otx1', 'Six4', 'E2f3', 'Neurog1', 'Hmga1', 'Dmrt3',
       'Dmrt2', 'Rcor2', 'Lhx9', 'Plagl2', 'Nfatc2', 'Sall4', 'Nr4a2',
       'Lef1', 'E2f2', 'Gbx1', 'Fezf1', 'Nfe2l3', 'Tcf7l1', 'Zfp599',
       'Smad3', 'Sox3'], dtype=object)

## Calculate TF triplet synergy

In [88]:
"""
Greedy O-information maximization starting with triplets.
"""

import numpy as np
import pandas as pd
from scipy.stats import entropy
from itertools import combinations
from tqdm import tqdm
from sklearn.metrics import mutual_info_score


class GreedyTFCombinationFinder:
    """Find optimal TF combinations starting with triplets."""
    
    def __init__(self, grn_df, counts_file, metadata_file, deseq_file, stage, n_bins=3):
        """
        Args:
            grn_df: GRN DataFrame with [TF, target, ...]
            counts_file, metadata_file, deseq_file: Data files
            stage: 'E14' or 'E18'
            n_bins: Number of bins for discretization
        """
        self.grn = grn_df
        self.stage = stage
        self.n_bins = n_bins
        
        # Load expression data
        counts = pd.read_csv(counts_file, sep=';', header=0).iloc[:, 1:]
        counts = counts.set_index(counts.columns[0])
        
        # Map to symbols
        deseq = pd.read_csv(deseq_file, index_col=0)
        ensembl_to_symbol = dict(zip(deseq['gene_id'], deseq['symbol']))
        counts.index = counts.index.map(lambda x: ensembl_to_symbol.get(x, x))
        counts = counts[~counts.index.duplicated(keep='first')]
        
        metadata = pd.read_csv(metadata_file, sep=';')
        
        # Get stage samples
        mask = (metadata['Stage'] == stage) & (metadata['Marker'] == 'Progenitors')
        samples = metadata[mask]['SampleID'].tolist()
        
        # Normalize
        expr = counts[samples]
        cpm = expr.div(expr.sum(axis=0), axis=1) * 1e6
        self.expr = np.log2(cpm + 1)
        
        print(f"Loaded expression for {stage}: {self.expr.shape[0]} genes, {self.expr.shape[1]} samples")
    
    def _discretize(self, x):
        """Discretize continuous values into bins."""
        return pd.qcut(x, q=self.n_bins, labels=False, duplicates='drop')
    
    def _multi_information(self, *variables):
        """
        Calculate multi-information (total correlation) for n variables.
        MI(X1, X2, ..., Xn) = sum(H(Xi)) - H(X1, X2, ..., Xn)
        """
        # Individual entropies
        individual_entropies = 0
        for var in variables:
            var_disc = self._discretize(var)
            counts = pd.Series(var_disc).value_counts()
            probs = counts / counts.sum()
            individual_entropies += entropy(probs, base=2)
        
        # Joint entropy
        discretized = [self._discretize(var) for var in variables]
        joint_df = pd.DataFrame({f'v{i}': discretized[i] for i in range(len(variables))})
        joint_df['joint'] = joint_df.astype(str).agg('_'.join, axis=1)
        
        counts = joint_df['joint'].value_counts()
        probs = counts / counts.sum()
        joint_entropy = entropy(probs, base=2)
        
        multi_info = individual_entropies - joint_entropy
        return multi_info
    
    def _o_information(self, *variables):
        """
        Calculate O-information for n variables.
        
        For triplets: O = -I(X;Y;Z)
        
        Where I(X;Y;Z) = interaction information = I(X;Y) + I(X;Z) + I(Y;Z) - I(X,Y;Z)
        
        Interpretation:
        I(X;Y;Z) > 0 → Synergy → O < 0
        I(X;Y;Z) < 0 → Redundancy → O > 0
        
        BUT we want intuitive signs, so return I directly:
        I > 0: Synergy (TFs cooperate)
        I < 0: Redundancy (TFs overlap)
        """
        n = len(variables)
        
        if n == 3:
            x, y, z = variables
            
            # Pairwise MIs
            mi_xy = mutual_info_score(self._discretize(x), self._discretize(y))
            mi_xz = mutual_info_score(self._discretize(x), self._discretize(z))
            mi_yz = mutual_info_score(self._discretize(y), self._discretize(z))
            
            # Joint MI I(X,Y;Z)
            x_disc = self._discretize(x)
            y_disc = self._discretize(y)
            z_disc = self._discretize(z)
            
            x_str = pd.Series(x_disc).astype(str)
            y_str = pd.Series(y_disc).astype(str)
            xy_joint = x_str + '_' + y_str
            xy_mapping = {v: i for i, v in enumerate(xy_joint.unique())}
            xy_numeric = xy_joint.map(xy_mapping).values
            
            mi_xyz = mutual_info_score(xy_numeric, z_disc)
            
            # Interaction information (synergy measure)
            interaction_info = mi_xy + mi_xz + mi_yz - mi_xyz
            
            tc = self._multi_information(*variables)
            
            # Return I directly (positive = synergy)
            return interaction_info, tc

    def _mutual_info_raw(self, x, y):
        """Calculate MI without using sklearn (for consistency)."""
        x_disc = self._discretize(x)
        y_disc = self._discretize(y)
        return mutual_info_score(x_disc, y_disc)
    
    def find_top_triplets(self, min_shared_targets=1, top_n=10):
        """
        Find top TF triplets (TF1, TF2, Target) by O-information.
        
        Returns:
            DataFrame with top triplets
        """
        print(f"\n=== Finding Top TF Triplets ===")
        
        # Get TF-target relationships
        tf_targets = {}
        for _, row in self.grn.iterrows():
            tf = row['TF']
            target = row['target']
            if tf not in tf_targets:
                tf_targets[tf] = set()
            tf_targets[tf].add(target)
        
        self.tf_targets = tf_targets
        
        # Find TF pairs with shared targets
        results = []
        tfs = [tf for tf in tf_targets.keys() if tf in self.expr.index]
        
        print(f"Evaluating {len(list(combinations(tfs, 2)))} TF pairs...")
        
        for tf1, tf2 in tqdm(combinations(tfs, 2), desc="Finding triplets", total=len(list(combinations(tfs, 2)))):
            shared_targets = tf_targets[tf1] & tf_targets[tf2]
            if len(shared_targets) < min_shared_targets:
                continue
            
            # Evaluate each shared target
            for target in shared_targets:
                if target not in self.expr.index:
                    continue
                
                tf1_expr = self.expr.loc[tf1].values
                tf2_expr = self.expr.loc[tf2].values
                target_expr = self.expr.loc[target].values
                
                try:
                    o_info, tc = self._o_information(tf1_expr, tf2_expr, target_expr)
                    
                    results.append({
                        'TF1': tf1,
                        'TF2': tf2,
                        'target': target,
                        'o_info': o_info,
                        'tc': tc
                    })
                except Exception as e:
                    continue
        
        triplets_df = pd.DataFrame(results).sort_values('o_info', ascending=False)
        
        print(f"Found {len(triplets_df)} valid triplets")
        print(f"\nTop {top_n} triplets:")
        print(triplets_df.head(top_n))
        
        return triplets_df.head(top_n)
    
    def greedy_expansion(self, initial_triplet, remaining_tfs, max_order=5):
        """
        Greedily add TFs to an initial triplet.
        
        Args:
            initial_triplet: Dict with keys TF1, TF2, target
            remaining_tfs: List of TFs to try adding
            max_order: Maximum combination order (including target)
        
        Returns:
            List of results for each expansion step
        """
        tf1 = initial_triplet['TF1']
        tf2 = initial_triplet['TF2']
        target = initial_triplet['target']
        
        current_tfs = [tf1, tf2]
        
        results = []
        
        # Baseline: order-3 (TF1, TF2, Target)
        exprs = [self.expr.loc[tf1].values, self.expr.loc[tf2].values, self.expr.loc[target].values]
        baseline, tc = self._o_information(*exprs)
        
        results.append({
            'order': 3,
            'tfs': current_tfs.copy(),
            'target': target,
            'o_info': baseline,
            'tc': tc
        })
        
        print(f"\n  Baseline ({tf1}, {tf2}, {target}): O-info = {baseline:.4f}")
        
        # Iteratively add TFs
        for order in range(4, max_order + 1):
            best_tf = None
            best_o_info = baseline
            best_tc = tc
            
            # Try adding each remaining TF
            for candidate_tf in remaining_tfs:
                if candidate_tf in current_tfs:
                    continue
                if candidate_tf not in self.expr.index:
                    continue
                
                # Check if candidate regulates the target
                if target not in self.tf_targets.get(candidate_tf, set()):
                    continue
                
                # Calculate O-info with candidate
                exprs = [self.expr.loc[tf].values for tf in current_tfs + [candidate_tf]] + [self.expr.loc[target].values]
                
                try:
                    candidate_o_info, candidate_tc = self._o_information(*exprs)
                except:
                    continue
                
                if candidate_o_info > best_o_info:
                    best_o_info = candidate_o_info
                    best_tf = candidate_tf
                    best_tc = candidate_tc
            
            # Stop if no improvement
            if best_tf is None or best_o_info <= baseline:
                print(f"  Order {order}: No improvement, stopping")
                break
            
            # Add best TF
            current_tfs.append(best_tf)
            baseline = best_o_info
            tc = best_tc
            
            results.append({
                'order': order,
                'tfs': current_tfs.copy(),
                'target': target,
                'o_info': best_o_info,
                'tc': best_tc
            })
            
            print(f"  Order {order}: Added {best_tf}, O-info = {best_o_info:.4f}")
        
        return results
    
    def find_optimal_combinations(self, top_triplets=10, max_order=6):
        """
        Main pipeline: find top triplets, then greedily expand each.
        
        Returns:
            DataFrame with all expansion results
        """
        # Step 1: Find top triplets
        triplets_df = self.find_top_triplets(top_n=top_triplets)
        
        if len(triplets_df) == 0:
            print("No valid triplets found!")
            return pd.DataFrame()
        
        # Step 2: Get all TFs
        all_tfs = list(self.tf_targets.keys())
        
        # Step 3: Expand each top triplet
        all_results = []
        
        print(f"\n=== Greedy Expansion ===")
        for idx, row in triplets_df.iterrows():
            initial_triplet = {
                'TF1': row['TF1'],
                'TF2': row['TF2'],
                'target': row['target']
            }
            
            remaining_tfs = [tf for tf in all_tfs if tf not in [row['TF1'], row['TF2']]]
            
            print(f"\nExpanding triplet {idx+1}/{len(triplets_df)}: ({row['TF1']}, {row['TF2']}, {row['target']})")
            
            expansion = self.greedy_expansion(initial_triplet, remaining_tfs, max_order=max_order)
            
            for result in expansion:
                result['initial_triplet'] = f"{row['TF1']}+{row['TF2']}→{row['target']}"
                result['stage'] = self.stage
                all_results.append(result)
        
        results_df = pd.DataFrame(all_results)
        
        # Summary
        print(f"\n=== Summary ===")
        print(f"Total combinations found: {len(results_df)}")
        print(f"\nBest combinations by order:")
        for order in sorted(results_df['order'].unique()):
            best = results_df[results_df['order'] == order].sort_values('o_info', ascending=False).iloc[0]
            print(f"  Order {order}: {best['tfs']} → {best['target']} - O-info = {best['o_info']:.4f}")
        
        return results_df


# === USAGE ===
# e14_finder = GreedyTFCombinationFinder(
#     grn_df=e14_grn,
#     counts_file=paths['counts'],
#     metadata_file=paths['metadata'],
#     deseq_file=paths['deseq_output'],
#     stage='E14',
#     n_bins=3
# )
# 
# e14_combinations = e14_finder.find_optimal_combinations(top_triplets=10, max_order=6)
# e14_combinations.to_csv('e14_optimal_tf_combinations.csv', index=False)

In [92]:

# === USAGE ===
e14_finder = GreedyTFCombinationFinder(
    grn_df=e14_grn,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    stage='E14',
    n_bins=3
)

e14_combinations = e14_finder.find_optimal_combinations(top_triplets=10, max_order=5)
e14_combinations.to_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/e14_optimal_tf_combinations.csv', index=False)

# E18
e18_finder = GreedyTFCombinationFinder(
    grn_df=e18_grn,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    stage='E18',
    n_bins=3
)

e18_combinations = e18_finder.find_optimal_combinations(top_triplets=10, max_order=5)
e18_combinations.to_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/e18_optimal_tf_combinations.csv', index=False)

Loaded expression for E14: 28656 genes, 10 samples

=== Finding Top TF Triplets ===
Evaluating 55 TF pairs...


Finding triplets: 100%|██████████| 55/55 [00:01<00:00, 40.99it/s]


Found 48 valid triplets

Top 10 triplets:
        TF1      TF2  target    o_info        tc
46     Rorc    Nr4a2  Zfp599  1.555323  2.243856
45    Sall4    Nr4a2   Rcor2  1.555323  2.243856
2      E2f2     E2f3   Hmga1  1.346023  2.541901
1      E2f2     E2f3  Plagl2  1.346023  2.541901
22    Nhlh1  Neurog1   Rcor2  1.259719  2.266412
3      E2f2     E2f3   Hmga2  1.207394  2.066412
25    Nhlh2   Bcl11b    Six4  1.121089  1.790924
47  Neurog1    Nr4a2   Rcor2  0.930135  2.017390
44    Sall4  Neurog1   Rcor2  0.930135  2.017390
20    Nhlh1    Sall4   Dmrt3  0.896155  2.217390

=== Greedy Expansion ===

Expanding triplet 47/10: (Rorc, Nr4a2, Zfp599)

  Baseline (Rorc, Nr4a2, Zfp599): O-info = 1.5553
  Order 4: No improvement, stopping

Expanding triplet 46/10: (Sall4, Nr4a2, Rcor2)

  Baseline (Sall4, Nr4a2, Rcor2): O-info = 1.5553
  Order 4: No improvement, stopping

Expanding triplet 3/10: (E2f2, E2f3, Hmga1)

  Baseline (E2f2, E2f3, Hmga1): O-info = 1.3460
  Order 4: No improvement, st

Finding triplets: 100%|██████████| 153/153 [00:01<00:00, 87.28it/s] 


Found 62 valid triplets

Top 10 triplets:
      TF1    TF2 target    o_info        tc
42  Foxf2  Foxq1   Etv4  1.761912  2.541901
37  Nr4a1    Fos  Ppara  1.536978  2.541901
61   Hey2  Rreb1   Rora  1.484653  2.341901
59   Gli1   Hey2  Rreb1  1.346023  2.492879
6   Klf15   Hey2   Rora  1.309333  2.190924
33   Klf9   Hey2   Rora  1.309333  2.190924
34   Klf9   Hey2  Foxo1  1.309333  2.190924
35   Klf9  Rreb1   Rora  1.241374  2.141901
8   Klf15  Rreb1   Rora  1.241374  2.141901
2   Klf15   Klf9   Rora  1.170704  2.390924

=== Greedy Expansion ===

Expanding triplet 43/10: (Foxf2, Foxq1, Etv4)

  Baseline (Foxf2, Foxq1, Etv4): O-info = 1.7619
  Order 4: No improvement, stopping

Expanding triplet 38/10: (Nr4a1, Fos, Ppara)

  Baseline (Nr4a1, Fos, Ppara): O-info = 1.5370
  Order 4: No improvement, stopping

Expanding triplet 62/10: (Hey2, Rreb1, Rora)

  Baseline (Hey2, Rreb1, Rora): O-info = 1.4847
  Order 4: No improvement, stopping

Expanding triplet 60/10: (Gli1, Hey2, Rreb1)

  Base

In [90]:
e14_combinations


,order,tfs,target,o_info,tc,initial_triplet,stage
0,3,"[Rorc, Nr4a2]",Zfp599,1.555323,2.243856,Rorc+Nr4a2→Zfp599,E14
1,3,"[Sall4, Nr4a2]",Rcor2,1.555323,2.243856,Sall4+Nr4a2→Rcor2,E14
2,3,"[E2f2, E2f3]",Hmga1,1.346023,2.541901,E2f2+E2f3→Hmga1,E14
3,3,"[E2f2, E2f3]",Plagl2,1.346023,2.541901,E2f2+E2f3→Plagl2,E14
4,3,"[Nhlh1, Neurog1]",Rcor2,1.259719,2.266412,Nhlh1+Neurog1→Rcor2,E14
5,3,"[E2f2, E2f3]",Hmga2,1.207394,2.066412,E2f2+E2f3→Hmga2,E14
6,3,"[Nhlh2, Bcl11b]",Six4,1.121089,1.790924,Nhlh2+Bcl11b→Six4,E14
7,3,"[Neurog1, Nr4a2]",Rcor2,0.930135,2.017390,Neurog1+Nr4a2→Rcor2,E14
8,3,"[Sall4, Neurog1]",Rcor2,0.930135,2.017390,Sall4+Neurog1→Rcor2,E14
9,3,"[Nhlh1, Sall4]",Dmrt3,0.896155,2.217390,Nhlh1+Sall4→Dmrt3,E14


In [91]:
e18_combinations

,order,tfs,target,o_info,tc,initial_triplet,stage
0,3,"[Foxf2, Foxq1]",Etv4,1.761912,2.541901,Foxf2+Foxq1→Etv4,E18
1,3,"[Nr4a1, Fos]",Ppara,1.536978,2.541901,Nr4a1+Fos→Ppara,E18
2,3,"[Hey2, Rreb1]",Rora,1.484653,2.341901,Hey2+Rreb1→Rora,E18
3,3,"[Gli1, Hey2]",Rreb1,1.346023,2.492879,Gli1+Hey2→Rreb1,E18
4,3,"[Klf15, Hey2]",Rora,1.309333,2.190924,Klf15+Hey2→Rora,E18
5,3,"[Klf9, Hey2]",Rora,1.309333,2.190924,Klf9+Hey2→Rora,E18
6,3,"[Klf9, Hey2]",Foxo1,1.309333,2.190924,Klf9+Hey2→Foxo1,E18
7,3,"[Klf9, Rreb1]",Rora,1.241374,2.141901,Klf9+Rreb1→Rora,E18
8,3,"[Klf15, Rreb1]",Rora,1.241374,2.141901,Klf15+Rreb1→Rora,E18
9,3,"[Klf15, Klf9]",Rora,1.170704,2.390924,Klf15+Klf9→Rora,E18
